# 06 Explanation Model: why prices differ across ZIP codes

**Purpose (professor feedback, group option A):** explain price differences across locations with ZIP-level crime,
school and demographic data, plus **year fixed effects**. ZIP fixed effects are left out here on purpose: a variable with one
value per ZIP would be absorbed by them. The website keeps its own prediction model (house features + ZIP and year fixed effects).

| ZIP-level variable | Source | Period |
|---|---|---|
| Median household income, % owner occupied, % bachelor's or higher, poverty rate, population | Census ACS 2024 5-year, by ZCTA (via the Census Reporter API, no key needed) | 2020 to 2024 |
| Violent and property crime per 1,000 residents per year | Philadelphia Police incidents (`incidents_part1_part2`, Carto API) | 2024 to 2025 |
| Average elementary school PSSA ELA proficiency (% proficient or advanced, grades 3-8, from SPREE) of the ZIP's homes | School District of Philadelphia catchments + SPREE | SY 2024-25 |

Census reports ZCTAs, which approximate ZIP codes. Crime incidents are assigned to the ZIP of the nearest home in our data.
**Run in Colab** (Runtime > Run all). It needs the cleaned CSV *with coordinates* (`x_2272`, `y_2272`).

AI use note: code drafted with Claude (Claude Code) from the group's chosen datasets and setup; see the AI Use Log at the end of Group_Project_Modeling.ipynb.


## 0. Setup

In [1]:
import os, json, zipfile, re, urllib.request, urllib.parse
import numpy as np, pandas as pd
import statsmodels.formula.api as smf
try:
    import geopandas as gpd
except ImportError:
    import subprocess, sys; subprocess.run([sys.executable, "-m", "pip", "install", "-q", "geopandas"], check=True); import geopandas as gpd
from scipy.spatial import cKDTree
from pyproj import Transformer

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
EXT = "data/external"; os.makedirs(EXT, exist_ok=True); os.makedirs("data/processed", exist_ok=True); os.makedirs("figures", exist_ok=True)
CLEAN_PATH = "data/processed/philly_sf_sales_clean.csv"
CLEAN_URL = "https://raw.githubusercontent.com/HyeseongO/philly-home-price-predictor/main/data/philly_sf_sales_clean.csv"
src = CLEAN_PATH if os.path.exists(CLEAN_PATH) else CLEAN_URL
df = pd.read_csv(src, low_memory=False, dtype={"zip_code": str, "parcel_number": str})
if "x_2272" not in df.columns:
    from google.colab import files
    print("This file has no coordinates. Upload the NEW philly_sf_sales_clean.csv (with x_2272 / y_2272).")
    name = list(files.upload().keys())[0]; os.makedirs("data/processed", exist_ok=True); os.replace(name, CLEAN_PATH)
    df = pd.read_csv(CLEAN_PATH, low_memory=False, dtype={"zip_code": str, "parcel_number": str})
df = df[df["bundle_size"] == 1].copy()
ZIPS = sorted(df["zip_code"].dropna().unique())
print(df.shape, "|", len(ZIPS), "ZIP codes")

def fetch(url, dest, params=None):
    if os.path.exists(dest): print("using saved", dest); return dest
    if params: url = url + "?" + urllib.parse.urlencode(params)
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 (student project)"})
    with urllib.request.urlopen(req, timeout=600) as r, open(dest, "wb") as f: f.write(r.read())
    print("downloaded", dest); return dest

(29572, 80) | 46 ZIP codes


## 1. Census demographics by ZCTA (ACS 2020-2024)

In [2]:
ACS = {"B01003001": "pop", "B19013001": "income", "B25003001": "occ", "B25003002": "owner",
       "B15003001": "pop25", "B15003022": "ba", "B15003023": "ma", "B15003024": "prof", "B15003025": "phd",
       "B17001001": "pov_u", "B17001002": "pov"}
TABLES = sorted({col[:6] for col in ACS})
af = fetch("https://api.censusreporter.org/1.0/data/show/acs2024_5yr", f"{EXT}/acs5_2024_zcta_philly.json",
           params={"table_ids": ",".join(TABLES), "geo_ids": ",".join("86000US" + z for z in ZIPS)})
raw = json.load(open(af))["data"]
a = pd.DataFrame([{"zip_code": geo.replace("86000US", ""),
                   **{name: raw[geo][col[:6]]["estimate"].get(col) for col, name in ACS.items()}} for geo in raw])
a[list(ACS.values())] = a[list(ACS.values())].apply(pd.to_numeric, errors="coerce").mask(lambda d: d < 0)
zipf = pd.DataFrame({"zip_code": a["zip_code"], "zip_pop": a["pop"], "zip_income": a["income"],
                     "zip_pct_owner": a["owner"] / a["occ"], "zip_pct_bachelors": (a["ba"] + a["ma"] + a["prof"] + a["phd"]) / a["pop25"],
                     "zip_pct_poverty": a["pov"] / a["pov_u"]})
print("ZCTAs returned:", len(zipf), "| ZIPs in our data without a Census match:", sorted(set(ZIPS) - set(zipf.zip_code)))

downloaded data/external/acs5_2024_zcta_philly.json
ZCTAs returned: 46 | ZIPs in our data without a Census match: []


## 2. Crime per 1,000 residents per year, by ZIP (2024-2025)
Each incident goes to the ZIP of the nearest home in our data (within 2,000 ft); incidents farther from any home are dropped.
Violent = UCR 100-499 (homicide, rape, robbery, aggravated assault); property = UCR 500-799.

In [3]:
q = ("SELECT ucr_general, ST_Y(the_geom) AS lat, ST_X(the_geom) AS lng FROM incidents_part1_part2 "
     "WHERE dispatch_date >= '2024-01-01' AND dispatch_date < '2026-01-01'")
cf = fetch("https://phl.carto.com/api/v2/sql", f"{EXT}/crime_2024_2025.csv", params={"q": q, "format": "csv"})
crime = pd.read_csv(cf).dropna(subset=["lat", "lng"])
crime["ucr"] = pd.to_numeric(crime["ucr_general"], errors="coerce")
to2272 = Transformer.from_crs("EPSG:4326", "EPSG:2272", always_xy=True)
cx, cy = to2272.transform(crime["lng"].values, crime["lat"].values)
homes = df.dropna(subset=["x_2272", "y_2272", "zip_code"])
dist, idx = cKDTree(homes[["x_2272", "y_2272"]].values).query(np.c_[cx, cy])
crime["zip_code"] = np.where(dist <= 2000, homes["zip_code"].values[idx], None)
print(f"incidents: {len(crime):,} | assigned to a ZIP: {crime.zip_code.notna().sum():,}")
cnt = crime.dropna(subset=["zip_code"]).groupby("zip_code").agg(
    violent=("ucr", lambda u: u.between(100, 499).sum()), prop=("ucr", lambda u: u.between(500, 799).sum())).reset_index()
zipf = zipf.merge(cnt, on="zip_code", how="left")
for k, col in [("violent", "zip_violent_per_1k"), ("prop", "zip_property_per_1k")]:
    zipf[col] = np.where(zipf["zip_pop"] >= 1000, zipf[k] / 2 / zipf["zip_pop"] * 1000, np.nan)
zipf[["zip_code", "zip_violent_per_1k", "zip_property_per_1k"]].describe().round(2)

downloaded data/external/crime_2024_2025.csv


incidents: 302,490 | assigned to a ZIP: 296,572


,zip_violent_per_1k,zip_property_per_1k
count,46.00,46.00
mean,7.64,44.96
std,5.24,25.51
min,1.20,19.46
25%,3.87,30.40
50%,5.91,40.17
75%,10.04,47.49
max,20.98,157.87


## 3. School quality by ZIP (elementary catchment PSSA ELA proficiency from SPREE, SY 2024-25)
Each home gets the PSSA ELA proficiency rate (% of grade 3-8 students proficient or advanced) of its elementary catchment school; the ZIP value is the average over its homes.
The district does not publish column names: this cell prints them. If the automatic choices are wrong, set the four names below and rerun.

In [4]:
SCHOOL_ID_COL = None; SPREE_ID_COL = None; SPREE_METRIC_COL = None; SPREE_METRIC = "PSSA ELA: % of Students Proficient or Advanced, Grades 3-8"; SPREE_VALUE_COL = None
cz = fetch("https://cdn.philasd.org/offices/performance/Open_Data/School_Information/School_Catchment/SDP_Catchment_2425.zip", f"{EXT}/SDP_Catchment_2425.zip")
ES_PATTERN = r"(^|[_\-\s])(es|elem\w*)([_\-\s.]|$)"
outer = zipfile.ZipFile(cz)
if not any(n.lower().endswith(".shp") for n in outer.namelist()):
    inner = next(n for n in outer.namelist() if re.search(ES_PATTERN, os.path.basename(n).lower()))
    cz = f"{EXT}/{os.path.basename(inner)}"
    if not os.path.exists(cz):
        with open(cz, "wb") as f: f.write(outer.read(inner))
shps = [n for n in zipfile.ZipFile(cz).namelist() if n.lower().endswith(".shp")]
es = next((n for n in shps if re.search(ES_PATTERN, os.path.basename(n).lower())), shps[0])
catch = gpd.read_file(f"zip://{cz}!{es}").to_crs("EPSG:2272")
SCHOOL_ID_COL = SCHOOL_ID_COL or next((c for c in catch.columns if any(k in c.upper() for k in ["ULCS", "ES_ID", "SCHOOL_ID", "SCH_ID", "SRC"])), catch.columns[0])
print("catchment file:", es, "| columns:", list(catch.columns), "| ID:", SCHOOL_ID_COL)
pts = gpd.GeoDataFrame(homes[["parcel_number", "zip_code"]], geometry=gpd.points_from_xy(homes["x_2272"], homes["y_2272"]), crs="EPSG:2272")
hs = gpd.sjoin(pts, catch[[SCHOOL_ID_COL, "geometry"]], how="left", predicate="within").drop_duplicates("parcel_number")
hs["school_id"] = hs[SCHOOL_ID_COL].astype(str).str.strip()
sp = pd.read_csv(fetch("https://cdn.philasd.org/offices/performance/Open_Data/School_Performance/SPREE/SPREE_2024-2025_School_Metric_Scores_Updated_April2026.csv",
                       f"{EXT}/SPREE_SY2024-2025.csv"), low_memory=False, encoding_errors="replace")
print("SPREE columns:", list(sp.columns))
low = {c.lower(): c for c in sp.columns}
SPREE_ID_COL = SPREE_ID_COL or next((low[k] for k in low if any(t in k for t in ["ulcs", "school_id", "school id", "src", "school code"])), None)
SPREE_METRIC_COL = SPREE_METRIC_COL or next((low[k] for k in low if "metric" in k and "score" not in k and "value" not in k), None)
SPREE_VALUE_COL = SPREE_VALUE_COL or next((low[k] for k in low if any(t in k for t in ["score", "value"])), None)
if SPREE_METRIC_COL: print("metrics:", sp[SPREE_METRIC_COL].value_counts().head(30).to_dict())
if SPREE_ID_COL and SPREE_VALUE_COL and (SPREE_METRIC_COL is None or SPREE_METRIC):
    s = sp if SPREE_METRIC_COL is None else sp[sp[SPREE_METRIC_COL] == SPREE_METRIC]
    score = s.assign(school_id=s[SPREE_ID_COL].astype(str).str.strip(), v=pd.to_numeric(s[SPREE_VALUE_COL], errors="coerce")).groupby("school_id")["v"].mean()
    hs["school_score"] = hs["school_id"].map(score)
    zipf = zipf.merge(hs.groupby("zip_code")["school_score"].mean().rename("zip_school_score").reset_index(), on="zip_code", how="left")
    print("homes with a school score:", int(hs.school_score.notna().sum()))
else:
    print("School score not merged yet: set SPREE_METRIC (and any wrong column name) from the lists above, then rerun this cell.")
zipf.to_csv("data/processed/zip_features.csv", index=False)

downloaded data/external/SDP_Catchment_2425.zip
catchment file: Catchments_ES_2024/Catchments_ES_2024.shp | columns: ['OBJECTID', 'ES_ID', 'ES_Name', 'MS_ID', 'MS_Name', 'HS_ID', 'HS_Name', 'GR_ID_K', 'GR_ID_01', 'GR_ID_02', 'GR_ID_03', 'GR_ID_04', 'GR_ID_05', 'GR_ID_06', 'GR_ID_07', 'GR_ID_08', 'GR_ID_09', 'GR_ID_10', 'GR_ID_11', 'GR_ID_12', 'ES_Grade', 'MS_Grade', 'HS_Grade', 'Shape_Leng', 'Shape_Area', 'geometry'] | ID: ES_ID


downloaded data/external/SPREE_SY2024-2025.csv
SPREE columns: ['accountability_year', 'school_id', 'school_name', 'report_type', 'metric', 'numerator', 'denominator', 'metric_score', 'score_previous', 'score_change', 'tier']
metrics: {'Admission Type': 296, '% of Students Attending Less Than 80% of Instructional Days': 296, '% of Students Attending 80-85% of Instructional Days': 296, '% of Students Attending 85-90% of Instructional Days': 296, '% of Students Attending at Least 90% of Instructional Days': 296, '% of Students Attending 90-95% of Instructional Days': 296, '% of Students Attending at Least 95% of Instructional Days': 296, 'Number of Dropouts - Last School of Touch (Grades 7-12)': 296, '# of Students Served Over the Full Year': 296, '% of Students Who Are American Indian/Alaskan Native': 296, '% of Students Who Are Asian': 296, '% of Students Who Are Black/African American': 296, '% of Students Who Are Economically Disadvantaged': 296, '% of Students Who Are English Learner

## 4. Explanation regression
log(sale price) on house features, ZIP-level neighborhood variables and **year fixed effects** (no ZIP fixed effects).
Standard errors are clustered by ZIP, because homes in the same ZIP share the same neighborhood values; with about 46 ZIPs,
the neighborhood coefficients rest on 46 groups, so treat their precision with care.
Units: income per $10,000; shares per 10 percentage points; crime per 1 incident per 1,000 residents; school score per 10 points.

In [5]:
d = df.merge(zipf, on="zip_code", how="left")
d["log_price"] = np.log(d["sale_price"]); d["log_area"] = np.log(d["total_livable_area"].where(d["total_livable_area"] > 0))
d["income_10k"] = d["zip_income"] / 10_000
for c in ["zip_pct_owner", "zip_pct_bachelors", "zip_pct_poverty"]: d[c + "_10pt"] = d[c] * 10
if "zip_school_score" in d: d["school_10pt"] = d["zip_school_score"] / 10
for c in ["interior_condition", "exterior_condition", "sale_year"]: d[c] = d[c].astype("Int64").astype(str).replace("<NA>", np.nan)
d["central_air"] = d["central_air"].fillna("Unknown")
nb = ["income_10k", "zip_pct_owner_10pt", "zip_pct_bachelors_10pt", "zip_pct_poverty_10pt", "zip_violent_per_1k", "zip_property_per_1k"]
if "school_10pt" in d: nb.append("school_10pt")
house = ["log_area", "number_of_bedrooms", "number_of_bathrooms", "year_built", "number_stories", "garage_spaces"]
need = ["log_price"] + house + nb + ["interior_condition", "exterior_condition", "central_air", "sale_year", "zip_code"]
dd = d.dropna(subset=need).copy()
both = dd[["interior_condition", "exterior_condition"]]
fold = [l for l in sorted(both.exterior_condition.unique()) if l != "4" and (both.exterior_condition.eq(l) == both.interior_condition.eq(l)).all()]
dd["exterior_condition"] = dd["exterior_condition"].replace({l: "4" for l in fold})
f = ("log_price ~ " + " + ".join(house + nb) + " + C(interior_condition, Treatment('4')) + C(exterior_condition, Treatment('4'))"
     " + C(central_air) + C(sale_year)")
m = smf.ols(f, data=dd).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(dd["zip_code"])[0]})
print(f"homes used: {len(dd):,} of {len(d):,} | ZIP clusters: {dd.zip_code.nunique()} | R2: {m.rsquared:.3f} | folded exterior codes: {fold}")
coef = pd.DataFrame({"coefficient": m.params, "std_error (ZIP clustered)": m.bse, "p_value": m.pvalues,
                     "approx_%_change_in_price": (np.exp(m.params) - 1) * 100})
coef.loc["Intercept", "approx_%_change_in_price"] = np.nan
coef.loc["log_area", "approx_%_change_in_price"] = np.nan
coef.to_csv("figures/explanation_model_coefficients.csv")
coef.loc[[i for i in coef.index if i in nb or i.startswith("C(sale_year)") or i == "log_area"]].round(4)

homes used: 27,998 of 29,572 | ZIP clusters: 46 | R2: 0.727 | folded exterior codes: ['1']


,coefficient,std_error (ZIP clustered),p_value,approx_%_change_in_price
C(sale_year)[T.2025],0.0131,0.0060,0.0296,1.3170
C(sale_year)[T.2026],0.0741,0.0106,0.0000,7.6939
log_area,0.7763,0.0396,0.0000,NaN
income_10k,0.0685,0.0204,0.0008,7.0848
zip_pct_owner_10pt,-0.0272,0.0251,0.2778,-2.6849
zip_pct_bachelors_10pt,-0.0160,0.0262,0.5414,-1.5857
zip_pct_poverty_10pt,0.0452,0.0315,0.1519,4.6205
zip_violent_per_1k,-0.0432,0.0061,0.0000,-4.2270
zip_property_per_1k,0.0058,0.0012,0.0000,0.5769
school_10pt,0.0694,0.0182,0.0001,7.1871


### How to read it
A coefficient of 0.05 on a variable means a one-unit increase goes with about 5% higher price, holding the other inputs fixed
(for log_area: 1% more living area goes with about coefficient % higher price). These are associations, not causal effects.